# Compare sequence sets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/compare_sequence_sets.ipynb)

Compare designed or experimental IDRs with a reference set using composition, embedding neighbors,
and optional SAE features and likelihood. Outputs: comparison tables and saved distribution plots.

Run cells from top to bottom. In Colab select **Runtime → Change runtime type → GPU**.
A GPU is recommended; CPU works but is slower. Runtime and peak memory depend on sequence
length, model, and hardware; timings are printed below rather than promising a fixed runtime.
First use downloads model weights. Outputs are written under `OUT_DIR`; rerunning replaces
files with the same names. Download that folder from Colab before ending the session.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git@v1"])

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record, read_fasta, parse_idr_header
print("Python:", sys.version.split()[0])
import idiom
print("IDiom:", idiom.__file__)


# Locate the companion helper in a clone, or download it for standalone Colab use.
helper_dir = next((p for p in (Path.cwd(), Path.cwd() / "cookbook/notebooks")
                   if (p / "workflow_utils.py").is_file()), None)
if helper_dir is None:
    from urllib.request import urlretrieve
    helper_dir = Path(".idiom_notebook_helpers")
    helper_dir.mkdir(exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/rotskoff-group/idiom/main/"
                "cookbook/notebooks/workflow_utils.py", helper_dir / "workflow_utils.py")
sys.path.insert(0, str(helper_dir.resolve()))
from workflow_utils import (AA, DEMO, load_inputs, idr_sequence, isolated, check_context,
                            summaries, write_fasta, save_run)


## Inputs and validation

Use `INPUT_MODE="idr"` for FASTA records that are **already isolated IDRs** (ordinary headers
are accepted). Use `INPUT_MODE="annotated"` for full proteins: the first header token must
end in `_IDR_x-y`, with **1-based inclusive** coordinates. This notebook does not predict IDR
boundaries. Python slices use 0-based, end-exclusive coordinates.

`INPUT_FASTA=None` uses six small illustrative sequences, not experimentally labeled examples.
Set a local path to analyze your own file (upload it using the Colab Files pane).
The audit table reports rejected records and records outside the sample limit. Empty sequences,
noncanonical residues, and invalid annotations are not silently repaired. Repeated accessions
remain distinct through `record_id`; duplicate IDR sequences are reported for your review.


In [ ]:
QUERY_FASTA = None # Demo: first three illustrative IDRs
REFERENCE_FASTA = None # Demo: last three illustrative IDRs
QUERY_MODE = "idr"
REFERENCE_MODE = "idr"
MAX_RECORDS = 32 # Per set; increase deliberately
MODEL = "jxliu2/idiom-20M"
LAYER = 5
DEVICE = "auto"
RUN_SAE = False # Optional: loads the 300M SAE host model
RUN_PERPLEXITY = False
SAE = "jxliu2/idiomsae-300M-L18-k32"
BATCH_SIZE = 2
OUT_DIR = Path("comparison_outputs")


In [ ]:
started = time.perf_counter()


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
query, qa = load_inputs(QUERY_FASTA, QUERY_MODE, MAX_RECORDS)
reference, ra = load_inputs(REFERENCE_FASTA, REFERENCE_MODE, MAX_RECORDS)
if QUERY_FASTA is None:
    query = query[:3]
if REFERENCE_FASTA is None:
    reference = reference[3:] if MAX_RECORDS is None or MAX_RECORDS > 3 else reference
# Prefix IDs so the two files cannot collide.
for prefix, recs, audit in [("query", query, qa), ("reference", reference, ra)]:
    audit.loc[(audit.status == "accepted") & ~audit.record_id.isin([r.accession for r in recs]), "status"] = "outside demo subset"
    audit["record_id"] = prefix + "_" + audit.record_id
query = [Record("query_" + r.accession, r.full_seq, r.idr_start, r.idr_end) for r in query]
reference = [Record("reference_" + r.accession, r.full_seq, r.idr_start, r.idr_end) for r in reference]
audit = pd.concat([qa, ra], ignore_index=True)
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
display(audit.groupby("status").size().rename("records"))
if not query or not reference:
    raise ValueError("Both sets need at least one accepted record.")
summary = pd.concat([summaries(query, qa).assign(group="query"),
                     summaries(reference, ra).assign(group="reference")], ignore_index=True)
summary.to_csv(OUT_DIR / "sequence_summary.csv", index=False)
shared = set(map(idr_sequence, query)) & set(map(idr_sequence, reference))
print(f"{len(shared)} unique IDR sequences occur in both sets; retained and flagged in neighbor results.")
print(summary.groupby("group").sequence.apply(lambda s: int(s.duplicated().sum())).rename("within-set duplicates"))


## Compare length and composition

These are descriptive comparisons. Duplicates, homologous sequences, species, and experimental
selection can influence differences. The charge proxy counts K/R as +1 and D/E as −1; it is
not a pH-dependent charge calculation. No statistical significance is inferred from these plots.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3), constrained_layout=True)
for ax, column in zip(axes, ["length", "charged_fraction", "net_charge_per_residue"]):
    bins = np.histogram_bin_edges(summary[column], bins=12)
    for group, frame in summary.groupby("group"):
        ax.hist(frame[column], bins=bins, density=True, histtype="step", label=group)
    ax.set(xlabel=column.replace("_", " "), ylabel="Density")
axes[0].legend()
fig.savefig(OUT_DIR / "sequence_properties.png", dpi=160)
plt.show()
group_means = summary.groupby("group")[[f"fraction_{a}" for a in AA]].mean().T
group_means.to_csv(OUT_DIR / "mean_composition.csv")
ax = group_means.plot.bar(figsize=(10, 3), ylabel="Mean per-sequence fraction")
ax.figure.tight_layout()
ax.figure.savefig(OUT_DIR / "composition.png", dpi=160)
plt.show()


## Find each query's closest reference

Both sets are embedded as isolated IDRs with identical settings. Cosine similarity is measured
in the full embedding space. A close neighbor does not establish shared function or novelty;
exact matches are explicitly marked. This matrix requires memory proportional to the product
of the set sizes.


In [ ]:
model = IDiom.from_pretrained(MODEL, device=DEVICE)
if not 0 <= LAYER < model.model.cfg.n_layers:
    raise ValueError("LAYER is outside this model's transformer blocks.")
all_records = isolated(query + reference)
check_context(all_records, model.model.cfg.max_seq_len)
values, _ = model.embed(all_records, layers=[LAYER], pool="mean")[LAYER]
np.save(OUT_DIR / "embeddings.npy", values)
summary[["record_id", "accession", "group"]].to_csv(OUT_DIR / "embedding_index.csv", index=False)
unit = values / np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1e-12)
cosine = unit[:len(query)] @ unit[len(query):].T
rows = []
for i, r in enumerate(query):
    j = int(cosine[i].argmax())
    rows.append(dict(query=r.accession, reference=reference[j].accession,
                     cosine_similarity=float(cosine[i, j]),
                     identical_idr=idr_sequence(r) == idr_sequence(reference[j])))
neighbors = pd.DataFrame(rows)
neighbors.to_csv(OUT_DIR / "nearest_references.csv", index=False)
display(neighbors)


## Optional: aggregate model likelihood

Enable `RUN_PERPLEXITY` to score both sets under the same isolated-IDR convention.


In [ ]:
if RUN_PERPLEXITY:
    from idiom.utils.perplexity import perplexity
    scores = []
    for label, recs in [("query", query), ("reference", reference)]:
        path = OUT_DIR / f"{label}_idrs.fasta"
        write_fasta(isolated(recs), path)
        score = perplexity(model.model, str(path), tokenizer=model.tok, device=model.device,
                           max_len=model.model.cfg.max_seq_len, prompted_prob=0.0,
                           completion_only=True, batch_size=BATCH_SIZE, num_workers=0, seed=0)
        if not score["n_tokens"]:
            raise ValueError(f"No tokens scored for {label}.")
        scores.append(dict(group=label, **score))
    scores = pd.DataFrame(scores)
    scores.to_csv(OUT_DIR / "aggregate_perplexity.csv", index=False)
    display(scores)
else:
    print("Aggregate perplexity disabled; set RUN_PERPLEXITY=True to calculate it.")


The optional likelihood cell above reports **token-weighted aggregate** NLL (nats), perplexity,
and scored token count for each set. It scores isolated IDRs plus STOP with no flanking context,
using identical conventions. These are not per-sequence rankings or quality/function scores.
Different lengths and composition affect the comparison; self-generated samples can naturally
score well under the generating model.


## Optional: compare SAE feature prevalence

Enable `RUN_SAE` to compare the fraction of sequences with any positive activation for each
feature. This uses sparse datasets and the SAE's own host model, independently of the embedding
model above. Differences are descriptive; use the enrichment notebook for length matching and
multiple-testing correction. Models are released from GPU memory before loading the SAE.


In [ ]:
if RUN_SAE:
    import gc
    import torch
    from idiom.sae.features.enrichment import feature_counts
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    sae = IDiomSAE.from_pretrained(SAE, device=DEVICE)
    if sae.fim_mode != "unprompted" or sae.region != "idr":
        raise ValueError("Use an unprompted IDR SAE.")
    check_context(all_records, sae.model.cfg.max_seq_len)
    qpath = sae.build_feature_dataset(query, OUT_DIR / "query_features", batch_size=BATCH_SIZE)
    rpath = sae.build_feature_dataset(reference, OUT_DIR / "reference_features", batch_size=BATCH_SIZE)
    qc, nq = feature_counts(qpath)
    rc, nr = feature_counts(rpath)
    features = pd.DataFrame(dict(feature_id=np.arange(len(qc)), query_prevalence=qc / nq,
                                 reference_prevalence=rc / nr))
    features["difference"] = features.query_prevalence - features.reference_prevalence
    features = features.sort_values("difference", key=lambda s: s.abs(), ascending=False)
    features.to_csv(OUT_DIR / "feature_prevalence.csv", index=False)
    display(features.head(20))
save_run(OUT_DIR, dict(query=QUERY_FASTA, reference=REFERENCE_FASTA, query_mode=QUERY_MODE,
                       reference_mode=REFERENCE_MODE, max_records=MAX_RECORDS, model=MODEL, layer=LAYER,
                       run_sae=RUN_SAE, sae=SAE, run_perplexity=RUN_PERPLEXITY, batch_size=BATCH_SIZE,
                       context="isolated IDRs", device=DEVICE), elapsed=time.perf_counter() - started)
print(f"Elapsed including model loads: {time.perf_counter() - started:.1f} s")


## Use the outputs

Keep `input_audit.csv` with your results: `record_id` connects exported rows to the original
accession and IDR span, even when accessions repeat. `run.json` records the settings and installed
package versions. Record exact model revisions separately when freezing a published analysis.

Next: [feature enrichment](feature_enrichment.ipynb) to test feature differences against
a deliberately chosen background, or [inspect features](inspect_sae_features.ipynb) in individual candidates.
